### Adapt from Mo's implementation

In [1]:
import pandas as pd
import sqlite3
import re
import networkx as nx
import igraph as ig
import time
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import cm
#import pycairo

In [2]:
seed = 3
random.seed(seed)
np.random.seed(seed)
del_input = False
del_output = False

### Database Connection and Data Loading

In [28]:
def load_sqlite_database(sql_path):
    """
    Load metadata and connectivity data from SQLite database.
    
    Parameters:
    -----------
    sql_path : str
        Path to SQLite database
        
    Returns:
    --------
    tuple
        (metadata_df, edgelist_df, synapses_df)
    """
    conn = sqlite3.connect(sql_path)
    
    # List all tables (for verification)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("Tables in the database:", tables)
    
    # Load all tables
    meta_df = pd.read_sql_query("SELECT * FROM meta", conn)
    edgelist_df = pd.read_sql_query("SELECT * FROM edgelist_simple", conn)
    #synapses_df = pd.read_sql_query("SELECT * FROM synapses", conn)
    
    conn.close()
    return meta_df, edgelist_df

# TODO: customize 

def preprocess_metadata(all_meta, index_col='id'):
    """
    Perform initial preprocessing on the metadata DataFrame.
    
    Parameters:
    -----------
    all_meta : pd.DataFrame
        Raw metadata DataFrame
    index_col : str
        Name of the column to use as index
        
    Returns:
    --------
    pd.DataFrame
        Preprocessed metadata DataFrame
    """
    # Basic cleaning
    all_meta = all_meta[all_meta[index_col] != '']
    all_meta[index_col] = all_meta[index_col].values.astype(int)
    all_meta = all_meta.set_index(index_col, drop=False)
    all_meta = all_meta.replace('', np.nan)
    
    # Filter and clean side information
    all_meta = all_meta[all_meta['side'] != 'na']
    all_meta['side'] = all_meta['side'].replace('midline', 'center')
    
    # Remove duplicates and add VNC mask
    all_meta = all_meta[~all_meta.index.duplicated(keep='first')]
    #vnc_mask = all_meta['root_id'].isnull()
    #all_meta['is_vnc'] = vnc_mask
    
    # Filter out rows with null cell_type
    all_meta = all_meta[~all_meta['cell_type'].isnull()]
    
    # Add super_class classifications
    #all_meta.loc[all_meta['cell_class'] == 'ascending_neuron', 'super_class'] = 'ascending'
    #all_meta.loc[all_meta['cell_class'] == 'descending_neuron', 'super_class'] = 'descending'
    #all_meta.loc[all_meta['cell_class'] == 'sensory_ascending', 'super_class'] = 'sensory_ascending'
    
    return all_meta

# TODO: customize 

def clean_origin_fields(all_meta):
    """
    Clean and standardize the origin field in the metadata.
    Simplifies entries and handles multi-neuropil cases.
    
    Parameters:
    -----------
    all_meta : pd.DataFrame
        Metadata DataFrame
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with cleaned origin fields
    """
    def simplify_origin(val):
        if type(val) == float:  # Handle NaN
            return val
        elif val == 'multi':
            return np.nan
        elif val == None:
            return np.nan
        else:
            # Clean up naming conventions
            #print (val)
            val = val.replace('_L', '').replace('_R', '').replace('tct', 'Tct')
            val = val.split('.')[0]
            
            # Coarse categorization
            val = val.lower()
            if 'leg' in val or 'ltct' in val:
                return 'leg'
            elif 'ntct' in val or 'neck' in val:
                return 'neck'
            elif 'w' in val:
                return 'wing'
            elif 'ht' in val or 'ha' in val:
                return 'haltere'
            elif 'anm' in val or 'ab' in val:
                return 'abdomen'
            elif 'vac' in val:
                return 'vac'
            elif 'prothorax' in val:
                return 'ventral prothorax'
            elif val == 'ov':
                return 'ov'
            elif 'tct' in val:
                return 'other tectulum'
            else:
                return np.nan  # brain, notum, cv
            
    all_meta['origin'] = all_meta['origin'].apply(simplify_origin)
    return all_meta

def add_neurotransmitter_annotations(all_meta):
    """
    Add neurotransmitter valence annotations to the metadata.
    
    Parameters:
    -----------
    all_meta : pd.DataFrame
        Metadata DataFrame
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with added neurotransmitter annotations
    """
    # Create excitatory/inhibitory masks
    excitatory_mask = (all_meta['top_nt'] == 'acetylcholine').astype(int)
    inhibitory_mask = all_meta['top_nt'].isin(('glutamate', 'gaba')).astype(int)
    
    # Calculate neurotransmitter valence (-1 for inhibitory, +1 for excitatory)
    nt_valence = excitatory_mask - inhibitory_mask
    all_meta['nt_valence'] = nt_valence
    
    return all_meta


In [6]:
sql_path = '/n/data1/hms/neurobio/wilson/banc/connectivity/manc_1.2.1_data.sqlite'
meta_df, edgelist_df = load_sqlite_database(sql_path)

Tables in the database: [('meta',), ('synapses',), ('edgelist_simple',), ('edgelist',)]


#### preprocessing on metadata

In [17]:
meta_df.columns

Index(['bodyid', 'neuromore', 'total_outputs', 'total_inputs', 'axon_outputs',
       'dend_outputs', 'axon_inputs', 'dend_inputs', 'pd_outputs',
       'pn_outputs', 'pd_inputs', 'pn_inputs', 'total_outputs_density',
       'total_inputs_density', 'axon_outputs_density', 'dend_outputs_density',
       'axon_inputs_density', 'dend_inputs_density', 'total_length',
       'axon_length', 'dend_length', 'pd_length', 'pn_length', 'axon_width',
       'dend_width', 'pd_width', 'pn_width', 'segregation_index',
       'projection_score', 'nodes', 'cable_length', 'dataset', 'post', 'pre',
       'voxels', 'top_nt', 'top_nt_p', 'side', 'nerve', 'hemilineage',
       'cell_class', 'cell_sub_class', 'cell_type', 'type', 'name',
       'other_names', 'origin', 'modality', 'receptor_type', 'soma_location',
       'tosoma_location', 'root_location', 'soma', 'top_p', 'supervoxel_id',
       'known_nt', 'known_nt_source'],
      dtype='object')

In [27]:
meta_df.origin.unique()

array(['brain', 'LegNp_R.LegNp_L', 'LegNpT1_R.LegNpT1_L',
       'NTct_L.LegNpT1_L', 'LegNpT1_R.NTct_R', 'multi', 'LTct.IntTct',
       'ANm', 'LegNp_R', 'IntTct.LTct', 'IntTct.LegNpT1_L', 'haltere',
       'NTct_R.IntTct', 'notum', 'LegNpT1_R', 'IntTct.HTct_L',
       'LegNpT3_L', 'UTct_R.UTct_L', 'wing base', 'mVACT3_L',
       'IntTct.LegNpT1_R', 'HTct_R', 'NTct_R', 'LegNpT1_L',
       'prothoracic leg', 'LegNpT1_L.Ov_L', 'LegNpT1_L.IntTct', 'LegNp_L',
       'NTct_R.HTct_R', 'NTct_R.LegNpT1_R', 'LegNpT1_L.LegNpT1_R',
       'CV.LegNpT1_R', 'IntTct.HTct_R', 'IntTct', 'HTct_R.NTct_R',
       'abdomen', 'Ov_R', 'Ov_L', 'LegNpT3', 'LegNpT3_R', 'ANm_R',
       'LegNpT2_L', 'WTct_L', 'metathoracic leg', 'wing',
       'LegNpT2_R.LegNpT2_L', 'wing margin', 'Ov_L.LTct_L', 'LegNpT2_R',
       'LegNpT2_L.LegNpT3_L', 'ANm.LegNpT3_L', 'WTct_R.WTct_L',
       'LegNpT1_R.IntTct_R', 'ANm_L', 'HTct_L', 'HTct_R.HTct_L',
       'IntTct_R.NTct_R', 'LegNpT1', 'UTct_R', 'mesothoracic leg',
       'LegN

In [29]:
# metadata preprocessing
# meta_df = preprocess_metadata(meta_df, index_col='bodyid')
meta_df = clean_origin_fields(meta_df)

In [30]:
edgelist_df

,post,pre,count,norm,post_count,pre_count,post_top_nt,post_top_nt_p,pre_top_nt,pre_top_nt_p
0,10000,153296,3,0.000068,1432,1390,acetylcholine,NaN,gaba,0.811
1,10000,166304,2,0.000045,1432,4684,acetylcholine,NaN,glutamate,NaN
2,10000,166265,14,0.000316,1432,1051,acetylcholine,NaN,gaba,0.565
3,10000,164229,3,0.000068,1432,954,acetylcholine,NaN,gaba,0.615
4,10000,42164,7,0.000158,1432,538,acetylcholine,NaN,acetylcholine,0.514
...,...,...,...,...,...,...,...,...,...,...
5305349,99990,16647,1,0.000178,366,2826,acetylcholine,0.909,glutamate,0.409
5305350,99990,16959,9,0.001601,366,2345,acetylcholine,0.909,acetylcholine,0.857
5305351,99990,11798,4,0.000711,366,5898,acetylcholine,0.909,gaba,NaN
5305352,99990,12062,2,0.000356,366,1519,acetylcholine,0.909,acetylcholine,NaN


#### preprocessing on edgelist

In [ ]:
def get_ids(all_meta, permit=None, col=None):
    """
    Get IDs for neurons matching specified criteria.
    
    Parameters:
    -----------
    permit : list or None
        List of permitted values
    col : str or None
        Column to filter on
        
    Returns:
    --------
    list
        List of matching neuron IDs
    """
    if permit is None:
        return None
    mask = all_meta[col].isin(permit)
    ids = all_meta[mask].index.to_list()
    return ids

def get_elist(edgelist_df, meta_df,source_ids=None, target_ids=None, min_norm=0.):
    """
    Filter edge list based on source and target neurons.
    
    Parameters:
    -----------
    full_elist : pd.DataFrame
        Complete edge list with columns ['pre', 'post', 'norm', ...]
    source_ids : list or None
        IDs of source (presynaptic) neurons to include
    target_ids : list or None
        IDs of target (postsynaptic) neurons to include
    min_norm : float
        Minimum normalized connection strength
        
    Returns:
    --------
    pd.DataFrame
        Filtered edge list
    """
    if (source_ids is None) and (target_ids is None):
        raise ValueError('Must provide source_ids or target_ids')
    
    # Start with norm threshold
    elist = edgelist_df[edgelist_df['norm'] >= min_norm].copy()
    
    # Apply source filter if specified
    if source_ids is not None:
        elist = elist[elist['pre'].isin(source_ids)]
    
    # Apply target filter if specified
    if target_ids is not None:
        elist = elist[elist['post'].isin(target_ids)]
    
    # Ensure IDs are integers
    elist['pre'] = elist['pre'].astype(int)
    elist['post'] = elist['post'].astype(int)
    
    # Remove connections involving neurons not in metadata
    elist = elist[elist['pre'].isin(meta_df.index) & 
                  elist['post'].isin(meta_df.index)]
    
    return elist

# Example usage:
"""
# With full edge list loaded
full_elist = pd.read_csv('full_edgelist.csv')  # or however you load it

# Get connections from specific neurons
source_neurons = [1001, 1002, 1003]
connections = get_elist(full_elist, source_ids=source_neurons)

# Get connections to specific neurons
target_neurons = [2001, 2002, 2003]
connections = get_elist(full_elist, target_ids=target_neurons)

# Get connections between two sets of neurons
connections = get_elist(full_elist, 
                       source_ids=source_neurons,
                       target_ids=target_neurons)
"""

def add_laterality_to_elist(elist_df, metadata_df):
    """
    Add laterality information to edge list based on neuron positions.
    
    Parameters:
    -----------
    elist_df : pd.DataFrame
        Edge list containing pre/post neuron connections
    metadata_df : pd.DataFrame
        Metadata containing neuron information including 'side' column
        
    Returns:
    --------
    pd.DataFrame
        Edge list with added laterality columns:
        - laterality: categorical (ipsilateral, contralateral, center_center, inward, outward)
        - laterality_num: numerical (-1: contralateral, 0: center, 1: ipsilateral)
    """
    # Get side information for connected neurons
    pre_side = metadata_df.loc[elist_df['pre'], 'side'].values
    post_side = metadata_df.loc[elist_df['post'], 'side'].values
    
    # Initialize all connections as contralateral
    laterality = np.full(pre_side.shape, 'contralateral')
    
    # Define different types of connections
    laterality[(pre_side == post_side) & (pre_side == 'center')] = 'center_center'
    laterality[(pre_side == post_side) & (pre_side != 'center')] = 'ipsilateral'
    laterality[(pre_side != post_side) & (post_side == 'center')] = 'inward'
    laterality[(pre_side != post_side) & (pre_side == 'center')] = 'outward'

    # Create numerical representation
    laterality_num = (laterality == 'ipsilateral').astype(int) - \
                    (laterality == 'contralateral').astype(int)
    
    # Add laterality information to edge list
    elist_df = elist_df.assign(laterality=laterality, laterality_num=laterality_num)
    return elist_df

def add_metadata_to_elist(elist_df, metadata_df, columns=None):
    """
    Add metadata and laterality information to edge list.
    
    Parameters:
    -----------
    elist_df : pd.DataFrame
        Edge list containing pre/post neuron connections
    metadata_df : pd.DataFrame
        Metadata containing neuron information
    columns : list or None
        List of metadata columns to add. If None, uses default set
        
    Returns:
    --------
    pd.DataFrame
        Edge list with added metadata and laterality information
    """
    # Default columns if none specified
    if columns is None:
        columns = ['cell_type', 'super_class', 'nt_valence']
    
    # Add specified metadata columns for both pre- and post-synaptic neurons
    annotations = {}
    for col in columns:
        for partner in ['pre', 'post']:
            key = f'{partner}_{col}'
            value = metadata_df.loc[elist_df[partner], col].values
            annotations[key] = value
    
    # Add metadata columns
    elist_df = elist_df.assign(**annotations)
    
    # Add laterality information
    elist_df = add_laterality_to_elist(elist_df, metadata_df)
    
    return elist_df

# Example usage:
"""
# Assuming you have edgelist_df and metadata_df loaded

# Add just laterality
elist_with_laterality = add_laterality_to_elist(edgelist_df, metadata_df)

# Add metadata and laterality
columns_to_add = ['cell_type', 'neurotransmitter', 'brain_region']
enriched_elist = add_metadata_to_elist(edgelist_df, metadata_df, columns=columns_to_add)

# Check results
print(enriched_elist['laterality'].value_counts())
print(enriched_elist[['pre_cell_type', 'post_cell_type', 'laterality']].head())
"""

In [ ]:

def get_source_target_elists(all_meta, linker, source=None, target=None,
                           linker_col='cell_type', source_col='cell_type', target_col='cell_type',
                           source_min_norm=.01, target_min_norm=.01, exclude_overlap=False):
    """
    Get edge lists for both source and target connections.
    
    Parameters:
    -----------
    linker : list
        List of linking neuron types
    source/target : list or None
        Lists of source/target neuron types
    *_col : str
        Column names for different neuron types
    *_min_norm : float
        Minimum normalization thresholds
    exclude_overlap : bool
        Whether to exclude overlapping connections
        
    Returns:
    --------
    tuple
        (source_elist, target_elist)
    """
    linker_ids = get_ids(all_meta, linker, linker_col)
    source_ids = get_ids(all_meta, source, source_col)
    target_ids = get_ids(all_meta, target, target_col)

    conn = sqlite3.connect(sql_path)  # Note: sql_path should be defined elsewhere
    source_elist = get_elist(conn, source_ids, linker_ids, source_min_norm)
    target_elist = get_elist(conn, linker_ids, target_ids, target_min_norm)

    if exclude_overlap:
        pre_mask = ~source_elist['pre'].isin(linker_ids)
        if target_ids is not None:
            pre_mask = pre_mask & ~source_elist['pre'].isin(target_ids)
        source_elist = source_elist[pre_mask]
    
        post_mask = ~target_elist['post'].isin(linker_ids)
        if source_ids is not None:
            post_mask = post_mask & ~target_elist['post'].isin(source_ids)
        target_elist = target_elist[post_mask]

    extras = ['nt_valence', 'neuromere', 'cell_class', 'super_class', 'origin']
    source_elist = add_meta_to_elist(source_elist, all_meta, [source_col, linker_col] + extras)
    target_elist = add_meta_to_elist(target_elist, all_meta, [linker_col, target_col] + extras)

    return source_elist, target_elist

def filter_by_norm(matrix, min_norm=0):
    """
    Filter matrix based on normalized values.
    
    Parameters:
    -----------
    matrix : pd.DataFrame
        Input matrix to filter
    min_norm : float
        Minimum normalization threshold
        
    Returns:
    --------
    pd.DataFrame
        Filtered matrix
    """
    normalized = matrix.values / matrix.values.sum(-1, keepdims=True)
    mask = np.any(normalized >= min_norm, axis=0)
    matrix = matrix.iloc[:, mask]
    return matrix

### Embedding and clustering analysis based on profile[k]

In [ ]:
# =================================================================================
# Dimensionality Reduction and Clustering Functions
# =================================================================================

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, NMF
from sklearn.cluster import SpectralClustering
from sklearn.metrics.pairwise import cosine_similarity
import umap
import matplotlib.pyplot as plt

class DimensionalityReduction:
    """
    Class handling various dimensionality reduction techniques for neural connectivity data
    """
    
    @staticmethod
    def perform_pca(data, n_components=None):
        """
        Perform PCA on connectivity data.
        
        Parameters:
        -----------
        data : np.ndarray
            Input data matrix (n_samples, n_features)
        n_components : int, optional
            Number of components to keep
            
        Returns:
        --------
        dict
            Contains PCA model, projections, and explained variance info
        """
        # Initialize and fit PCA
        pca_model = PCA(n_components=n_components)
        projections = pca_model.fit_transform(data)
        
        return {
            'model': pca_model,
            'projections': projections,
            'explained_variance_ratio': pca_model.explained_variance_ratio_,
            'components': pca_model.components_
        }
    
    @staticmethod
    def perform_umap(data, n_neighbors=15, min_dist=0.1, n_components=2, 
                    random_state=None, metric='cosine'):
        """
        Perform UMAP dimensionality reduction.
        
        Parameters:
        -----------
        data : np.ndarray
            Input data matrix
        n_neighbors : int
            Size of local neighborhood
        min_dist : float
            Minimum distance between points in embedding
        
        Returns:
        --------
        dict
            Contains UMAP model and embeddings
        """
        umap_model = umap.UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=n_components,
            random_state=random_state,
            metric=metric
        )
        
        embeddings = umap_model.fit_transform(data)
        
        return {
            'model': umap_model,
            'embeddings': embeddings
        }
    
    @staticmethod
    def perform_nmf(data, n_components=16, random_state=None):
        """
        Perform Non-negative Matrix Factorization.
        
        Parameters:
        -----------
        data : np.ndarray
            Input data matrix (non-negative values)
        n_components : int
            Number of components
            
        Returns:
        --------
        dict
            Contains NMF model, projections, and components
        """
        nmf_model = NMF(
            n_components=n_components, 
            random_state=random_state
        )
        
        projections = nmf_model.fit_transform(data)
        
        return {
            'model': nmf_model,
            'projections': projections,
            'components': nmf_model.components_
        }

class Clustering:
    """
    Class handling clustering analyses
    """
    
    @staticmethod
    def perform_spectral_clustering(data, n_clusters=2, random_state=None):
        """
        Perform spectral clustering using cosine similarity.
        
        Parameters:
        -----------
        data : np.ndarray
            Input data matrix
        n_clusters : int
            Number of clusters to form
            
        Returns:
        --------
        dict
            Contains clustering model and labels
        """
        # Compute cosine similarity matrix
        data_normalized = data / np.linalg.norm(data, axis=-1, keepdims=True)
        affinity_matrix = data_normalized @ data_normalized.T
        
        # Perform clustering
        clustering = SpectralClustering(
            n_clusters=n_clusters,
            affinity='precomputed',
            random_state=random_state
        )
        
        labels = clustering.fit_predict(affinity_matrix)
        
        return {
            'model': clustering,
            'labels': labels,
            'affinity_matrix': affinity_matrix
        }

class Visualization:
    """
    Class handling visualization of dimensionality reduction and clustering results
    """
    
    @staticmethod
    def plot_explained_variance(pca_results, title=''):
        """Plot PCA explained variance"""
        cumulative_variance = np.cumsum(pca_results['explained_variance_ratio'])
        explained_variance = pca_results['explained_variance_ratio']
        
        fig, ax1 = plt.subplots(figsize=(8, 5))
        
        # Plot individual variance
        ax1.bar(range(len(explained_variance)), explained_variance, 
                alpha=0.6, color='g')
        ax1.set_xlabel('Number of Components')
        ax1.set_ylabel('Explained Variance per Component', color='g')
        ax1.tick_params(axis='y', labelcolor='g')
        
        # Plot cumulative variance
        ax2 = ax1.twinx()
        ax2.plot(range(len(cumulative_variance)), cumulative_variance, 
                color='b')
        ax2.set_ylabel('Cumulative Explained Variance', color='b')
        ax2.tick_params(axis='y', labelcolor='b')
        
        plt.title(title)
        return fig, (ax1, ax2)
    
    @staticmethod
    def plot_embedding(embedding, labels=None, title=''):
        """Plot 2D embedding with optional labels"""
        plt.figure(figsize=(10, 8))
        if labels is None:
            plt.scatter(embedding[:, 0], embedding[:, 1])
        else:
            plt.scatter(embedding[:, 0], embedding[:, 1], c=labels)
        plt.title(title)
        plt.colorbar()
        return plt.gcf()

# Example usage:
def analyze_connectivity_patterns(profiles, n_components=10):
    """
    Perform comprehensive analysis of connectivity patterns.
    
    Parameters:
    -----------
    profiles : dict
        Dictionary of connectivity profiles
    n_components : int
        Number of components for dimensionality reduction
        
    Returns:
    --------
    dict
        Results from all analyses
    """
    results = {}
    
    for key, data in profiles.items():
        results[key] = {
            'pca': DimensionalityReduction.perform_pca(data, n_components),
            'umap': DimensionalityReduction.perform_umap(data),
            'nmf': DimensionalityReduction.perform_nmf(data, n_components),
            'clustering': Clustering.perform_spectral_clustering(data)
        }
        
        # Generate visualizations
        Visualization.plot_explained_variance(
            results[key]['pca'], 
            f'PCA Explained Variance - {key}'
        )
        
        Visualization.plot_embedding(
            results[key]['umap']['embeddings'],
            results[key]['clustering']['labels'],
            f'UMAP Embedding - {key}'
        )
    
    return results

In [ ]:
# Analyze your connectivity profiles
results = analyze_connectivity_patterns(profiles)

# Access specific results
pca_results = results['descending_output']['pca']
umap_embedding = results['descending_output']['umap']['embeddings']
cluster_labels = results['descending_output']['clustering']['labels']

# Generate specific visualizations
Visualization.plot_embedding(
    umap_embedding,
    cluster_labels,
    'UMAP of Descending Neuron Outputs'
)